In [ ]:
!pip install flask pyngrok werkzeug -q
print(" Libraries installed!")

 Libraries installed!


In [ ]:
from pyngrok import ngrok
# ngrok.com → Dashboard → Copy "Your Authtoken"
ngrok.set_auth_token("2YOUR_TOKEN_HERE_123abc...")  # <-- YOUR TOKEN लिहा
print("Ngrok ready!")

Ngrok ready!


In [ ]:
from flask import Flask, render_template_string, request, redirect, url_for
from flask_sqlalchemy import SQLAlchemy
from flask_login import LoginManager, UserMixin, login_user, login_required, logout_user, current_user
from werkzeug.security import generate_password_hash, check_password_hash
from pyngrok import ngrok

# App setup
app = Flask(__name__)
app.config['SECRET_KEY'] = 'secret123'
app.config['SQLALCHEMY_DATABASE_URI'] = 'sqlite:///users.db'

db = SQLAlchemy(app)
login_manager = LoginManager()
login_manager.init_app(app)

# ------------------ DATABASE ------------------
class User(UserMixin, db.Model):
    id = db.Column(db.Integer, primary_key=True)
    email = db.Column(db.String(100), unique=True)
    password = db.Column(db.String(200))

class Activity(db.Model):
    id = db.Column(db.Integer, primary_key=True)
    user_id = db.Column(db.Integer)
    action = db.Column(db.String(200))

# ------------------ LOGIN ------------------
@login_manager.user_loader
def load_user(user_id):
    return User.query.get(int(user_id))

# ------------------ ROUTES ------------------

# Home
@app.route('/')
def home():
    return '<h1>Welcome</h1><a href="/login">Login</a> | <a href="/register">Register</a>'

# Register
@app.route('/register', methods=['GET','POST'])
def register():
    if request.method == 'POST':
        email = request.form['email']
        password = generate_password_hash(request.form['password'])

        user = User(email=email, password=password)
        db.session.add(user)
        db.session.commit()

        return redirect('/login')

    return '''
    <form method="post">
        Email: <input name="email"><br>
        Password: <input name="password" type="password"><br>
        <button>Register</button>
    </form>
    '''

# Login
@app.route('/login', methods=['GET','POST'])
def login():
    if request.method == 'POST':
        email = request.form['email']
        password = request.form['password']

        user = User.query.filter_by(email=email).first()

        if user and check_password_hash(user.password, password):
            login_user(user)

            # Activity log
            activity = Activity(user_id=user.id, action="Logged in")
            db.session.add(activity)
            db.session.commit()

            return redirect('/dashboard')

    return '''
    <form method="post">
        Email: <input name="email"><br>
        Password: <input name="password" type="password"><br>
        <button>Login</button>
    </form>
    '''

# Dashboard
@app.route('/dashboard')
@login_required
def dashboard():
    activities = Activity.query.filter_by(user_id=current_user.id).all()
    logs = "<br>".join([a.action for a in activities])

    return f'''
    <h1>Dashboard</h1>
    <p>Welcome {current_user.email}</p>
    <p>Your Activity:</p>
    {logs}
    <br><a href="/logout">Logout</a>
    '''

# Logout
@app.route('/logout')
@login_required
def logout():
    logout_user()
    return redirect('/')

# ------------------ RUN ------------------
db.create_all()

# NGROK
ngrok.set_auth_token("YOUR_NGROK_TOKEN")
public_url = ngrok.connect(5000)
print("Your App URL:", public_url)

app.run(port=5000)

RuntimeError: Working outside of application context.

This typically means that you attempted to use functionality that needed
the current application. To solve this, set up an application context
with app.app_context(). See the documentation for more information.

In [ ]:
from pyngrok import ngrok

ngrok.set_auth_token("3CNxw3Yk6dJ56VhjjzzvZi4lKSvJ_5WZV4uD2FwEeMAu5S8LiG")

In [ ]:
!pip install flask flask-login flask-sqlalchemy pyngrok werkzeug

In [ ]:
!pip install flask flask-login flask-sqlalchemy pyngrok werkzeug

In [ ]:
# Install
!pip install flask flask-login flask-sqlalchemy pyngrok werkzeug

from flask import Flask, request, redirect
from flask_sqlalchemy import SQLAlchemy
from flask_login import LoginManager, UserMixin, login_user, login_required, logout_user, current_user
from werkzeug.security import generate_password_hash, check_password_hash
from pyngrok import ngrok

# APP
app = Flask(__name__)
app.config['SECRET_KEY'] = 'secret123'
app.config['SQLALCHEMY_DATABASE_URI'] = 'sqlite:///users.db'

db = SQLAlchemy(app)
login_manager = LoginManager()
login_manager.init_app(app)

# DB
class User(UserMixin, db.Model):
    id = db.Column(db.Integer, primary_key=True)
    email = db.Column(db.String(100), unique=True)
    password = db.Column(db.String(200))

class Activity(db.Model):
    id = db.Column(db.Integer, primary_key=True)
    user_id = db.Column(db.Integer)
    action = db.Column(db.String(200))

@login_manager.user_loader
def load_user(user_id):
    return User.query.get(int(user_id))

# ROUTES
@app.route('/')
def home():
    return '<h2>Home</h2><a href="/register">Register</a> | <a href="/login">Login</a>'

@app.route('/register', methods=['GET','POST'])
def register():
    if request.method == 'POST':
        email = request.form['email']
        password = generate_password_hash(request.form['password'])

        user = User(email=email, password=password)
        db.session.add(user)
        db.session.commit()

        return redirect('/login')

    return '''
    <h2>Register</h2>
    <form method="post">
        Email: <input name="email"><br>
        Password: <input type="password" name="password"><br>
        <button>Register</button>
    </form>
    '''

@app.route('/login', methods=['GET','POST'])
def login():
    if request.method == 'POST':
        email = request.form['email']
        password = request.form['password']

        user = User.query.filter_by(email=email).first()

        if user and check_password_hash(user.password, password):
            login_user(user)

            activity = Activity(user_id=user.id, action="Logged in")
            db.session.add(activity)
            db.session.commit()

            return redirect('/dashboard')

    return '''
    <h2>Login</h2>
    <form method="post">
        Email: <input name="email"><br>
        Password: <input type="password" name="password"><br>
        <button>Login</button>
    </form>
    '''

@app.route('/dashboard')
@login_required
def dashboard():
    activities = Activity.query.filter_by(user_id=current_user.id).all()
    logs = "<br>".join([a.action for a in activities])

    return f'''
    <h2>Dashboard</h2>
    <p>Welcome: {current_user.email}</p>
    <p><b>Activity:</b></p>
    {logs}
    <br><br>
    <a href="/logout">Logout</a>
    '''

@app.route('/logout')
@login_required
def logout():
    logout_user()
    return redirect('/')

# RUN
with app.app_context():
    db.create_all()

ngrok.set_auth_token("3CNxw3Yk6dJ56VHjzzvZi4lKSvJ_5WZV4uD2FwEeMAu5S8LiG")

public_url = ngrok.connect(5000)
print("Your App URL:", public_url)

app.run(port=5000)

Your App URL: NgrokTunnel: "https://plastic-viable-majesty.ngrok-free.dev" -> "http://localhost:5000"
 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on http://127.0.0.1:5000
INFO:werkzeug:Press CTRL+C to quit
INFO:werkzeug:127.0.0.1 - - [15/Apr/2026 08:37:43] "GET / HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [15/Apr/2026 08:37:44] "GET /favicon.ico HTTP/1.1" 404 -
INFO:werkzeug:127.0.0.1 - - [15/Apr/2026 08:37:58] "GET /login HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [15/Apr/2026 08:38:32] "POST /login HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [15/Apr/2026 08:38:57] "POST /login HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [15/Apr/2026 08:39:25] "POST /login HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [15/Apr/2026 08:39:34] "GET /login HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [15/Apr/2026 08:39:36] "GET /register HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [15/Apr/2026 08:39:45] "POST /register HTTP/1.1" 302 -
INFO:werkzeug:127.0.0.1 - - [15/Apr/2026 08:39:45] "GET /login HTTP/1.1" 2